## Predictor Variable
<p align="justify">
Now that we have our water quality dataset, the next step is to gather the predictor variables from the <b>Landsat</b> and <b>TerraClimate</b> datasets. In this notebook, we demonstrate how to <b>load previously extracted satellite and climate data</b> from separate files, rather than performing the extraction directly, which allows for a smoother and faster experience. Participants can refer to the dedicated extraction notebooks—one for Landsat and another for TerraClimate—to understand how the data was retrieved and processed, and they can also generate their own output CSV files if needed. Using these pre-extracted CSV files, this notebook focuses on loading the predictor features and running the subsequent analysis and model training efficiently.
</p>
<p align="justify">
For more detailed guidance on the original data extraction process, you can review the <a href="https://planetarycomputer.microsoft.com/dataset/landsat-c2-l2#Example-Notebook">Landsat example notebook</a> and the <a href="https://planetarycomputer.microsoft.com/dataset/terraclimate#Example-Notebook">TerraClimate example notebook</a> available on the Planetary Computer portal.
</p>

<p align="justify">We have used selected spectral bands — SWIR22 (Shortwave Infrared 2), NIR (Near Infrared), Green, and SWIR16 (Shortwave Infrared 1) — and computed key spectral indices such as NDMI (Normalized Difference Moisture Index) and MNDWI (Modified Normalized Difference Water Index). These features capture surface moisture, vegetation, and water content characteristics that influence water quality variability. </p> <p align="justify"> In addition to Landsat features, we also incorporated the <b>Potential Evapotranspiration (PET)</b> variable from the <b>TerraClimate</b> dataset, which provides high-resolution global climate data. The PET feature captures the atmospheric demand for moisture, representing climatic conditions such as temperature, humidity, and radiation that influence surface water evaporation and thus affect water quality parameters. </p> <ul> <li>SWIR22 – Sensitive to surface moisture and turbidity variations in water bodies.</li> <li>NIR – Helps in identifying vegetation and suspended matter in water.</li> <li>Green – Useful for detecting water color and surface reflectance changes.</li> <li>SWIR16 – Provides information on surface dryness and sediment concentration.</li> <li>NDMI – Derived from NIR and SWIR16, indicates moisture and vegetation-water interaction.</li> <li>MNDWI – Derived from Green and SWIR22, effective for distinguishing open water areas and reducing built-up noise.</li> <li>PET – Extracted from the TerraClimate dataset, represents the potential evapotranspiration that influences hydrological and water quality dynamics.</li> </ul>

In [1]:
%load_ext autoreload
%autoreload 2

## Load data

In [2]:
import pandas as pd
Water_Quality_df = pd.read_csv('./data/original/water_quality_training_dataset.csv')
Water_Quality_df.head()


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.2.2 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/cvmfs/soft.computecanada.ca/easybuild/software/2023/x86-64-v4/Compiler/gcccore/ipykernel/2025a/lib/python3.11/site-packages/ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "/cvmfs/soft.computecanada.ca/easybuild/software/2023/x86-64-v4/Compiler/gcccore/ipykernel/2025a/lib/python3.11/site-packages/traitlets/config/application.py", line 1075, in launch_instance
    app.start()
  

AttributeError: _ARRAY_API not found

,Latitude,Longitude,Sample Date,Total Alkalinity,Electrical Conductance,Dissolved Reactive Phosphorus
0,-28.760833,17.730278,02-01-2011,128.912,555.0,10.0
1,-26.861111,28.884722,03-01-2011,74.720,162.9,163.0
2,-26.450000,28.085833,03-01-2011,89.254,573.0,80.0
3,-27.671111,27.236944,03-01-2011,82.000,203.6,101.0
4,-27.356667,27.286389,03-01-2011,56.100,145.1,151.0


In [3]:
landsat_train_features = pd.read_csv('./data/original/landsat_features_training.csv')
landsat_train_features.head()

,Latitude,Longitude,Sample Date,nir,green,swir16,swir22,NDMI,MNDWI
0,-28.760833,17.730278,02-01-2011,11190.0,11426.0,7687.5,7645.0,0.185538,0.195595
1,-26.861111,28.884722,03-01-2011,17658.5,9550.0,13746.5,10574.0,0.124566,-0.180134
2,-26.450000,28.085833,03-01-2011,15210.0,10720.0,17974.0,14201.0,-0.083293,-0.252805
3,-27.671111,27.236944,03-01-2011,14887.0,10943.0,13522.0,11403.0,0.048048,-0.105416
4,-27.356667,27.286389,03-01-2011,16828.5,9502.5,12665.5,9643.0,0.141147,-0.142683


In [4]:
Terraclimate_df = pd.read_csv('./data/original/terraclimate_features_training.csv')
Terraclimate_df.head()

,Latitude,Longitude,Sample Date,pet
0,-28.760833,17.730278,02-01-2011,174.2
1,-26.861111,28.884722,03-01-2011,124.1
2,-26.450000,28.085833,03-01-2011,127.5
3,-27.671111,27.236944,03-01-2011,129.7
4,-27.356667,27.286389,03-01-2011,129.2


In [5]:
from utils.pipeline import *
MERGE_KEYS = ['Latitude', 'Longitude', 'Sample Date']
wq_data = combine_two_datasets(Water_Quality_df, landsat_train_features, Terraclimate_df, keys=MERGE_KEYS)

ModuleNotFoundError: No module named 'google'

## Preprocess data

In [ ]:
wq_data = wq_data[['swir22','NDMI','MNDWI','pet', 'Total Alkalinity', 'Electrical Conductance', 'Dissolved Reactive Phosphorus']]

## Run pipeline


In [ ]:
X = wq_data.drop(columns=['Total Alkalinity', 'Electrical Conductance', 'Dissolved Reactive Phosphorus'])

y_TA = wq_data['Total Alkalinity']
y_EC = wq_data['Electrical Conductance']
y_DRP = wq_data['Dissolved Reactive Phosphorus']

model_TA, results_TA = run_pipeline(X, y_TA, "Total Alkalinity")
model_EC, results_EC = run_pipeline(X, y_EC, "Electrical Conductance")
model_DRP, results_DRP = run_pipeline(X, y_DRP, "Dissolved Reactive Phosphorus")

## Out-of-fold validation (OOF)

Like the reference code: train one model per fold, fill OOF predictions for the training set, and (optionally) average fold models for test predictions. This gives a more reliable estimate of generalization and can improve scores.

- **`pipeline_kind='simple'`** – benchmark-style: imputer + scaler + RandomForest (no PCA). Often yields higher test R² than the full PCA+XGBoost pipeline.
- **`pipeline_kind='full'`** – current pipeline: imputer + scaler + PCA + XGBoost.
- Pass **`X_test`** to get averaged test predictions from all fold models.

In [ ]:
from utils.pipeline import run_pipeline_oof

N_SPLITS = 5
SEED = 42

# OOF with simple (benchmark-style) pipeline – often better test R² than full pipeline
models_TA, oof_TA, results_TA_oof, _ = run_pipeline_oof(X, y_TA, n_splits=N_SPLITS, param_name="Total Alkalinity", pipeline_kind='simple', random_state=SEED)
models_EC, oof_EC, results_EC_oof, _ = run_pipeline_oof(X, y_EC, n_splits=N_SPLITS, param_name="Electrical Conductance", pipeline_kind='simple', random_state=SEED)
models_DRP, oof_DRP, results_DRP_oof, _ = run_pipeline_oof(X, y_DRP, n_splits=N_SPLITS, param_name="Dissolved Reactive Phosphorus", pipeline_kind='simple', random_state=SEED)

# Optional: with held-out X_test, pass it to get averaged test predictions
# models_TA, oof_TA, results_TA_oof, pred_test_TA = run_pipeline_oof(X, y_TA, X_test=X_test, n_splits=N_SPLITS, ...)